# MNIST: affine score, MLP and CNN

This notebook accompanies Exercise 6.2. All three machines produce ten scores and use the same cross-entropy loss. They differ in the structure imposed on the function $x\mapsto\Phi(x,p)$.

The first run uses a subset to remain fast on a CPU. At the end of the exercise, set `MODE_RAPIDE` to `False` to use the full dataset.

## Outline

1. [Loading and frequencies](#donnees-mnist)
2. [Affine score](#affine-mnist)
3. [MLP](#mlp-mnist)
4. [CNN and parameter sharing](#cnn-mnist)
5. [Comparaison](#comparaison-mnist)
6. [Confusion matrix and errors](#confusion-mnist)

In [ ]:
from time import perf_counter

import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import fetch_openml
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

In [ ]:
MODE_RAPIDE = True

if MODE_RAPIDE:
    N_APP_TOTAL = 12_000
    N_TEST = 2_000
    EPOQUES = 5
else:
    N_APP_TOTAL = 60_000
    N_TEST = 10_000
    EPOQUES = 10

TAILLE_LOT = 128

<a id="donnees-mnist"></a>
## 1. Loading and frequencies

Load `mnist_784` from OpenML. The first load requires an internet connection; subsequent runs use the local `scikit-learn` cache.

Normalise pixels to $[0,1]$, retain the official split between 60,000 training images and 10,000 test images, then construct a validation set using only the training data. In fast mode, perform stratified subsampling.

In [ ]:
# Complete this section.

All three machines will be trained with the same batches and loss. The following functions organise the experiment; machine structure remains visible in their respective classes.

In [ ]:
def perte(machine, x, z):
    scores = machine(x)
    return optax.softmax_cross_entropy_with_integer_labels(scores, z).mean()


@nnx.jit
def pas(machine, optimizer, x, z):
    valeur, gradient = nnx.value_and_grad(perte)(machine, x, z)
    optimizer.update(machine, gradient)
    return valeur


def entrainer(machine, images, z, *, alpha=1e-3, epoques=EPOQUES, graine=0):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    generateur = np.random.default_rng(graine)
    historique = []
    debut = perf_counter()
    for _ in range(epoques):
        permutation = generateur.permutation(z.size)
        pertes = []
        for debut_lot in range(0, z.size, TAILLE_LOT):
            indices = permutation[debut_lot:debut_lot + TAILLE_LOT]
            pertes.append(
                float(pas(machine, optimizer, jnp.asarray(images[indices]), jnp.asarray(z[indices])))
            )
        historique.append(float(np.mean(pertes)))
    return historique, perf_counter() - debut


def evaluer(machine, images, z, taille_lot=512):
    scores = []
    for debut_lot in range(0, z.size, taille_lot):
        scores.append(np.asarray(machine(jnp.asarray(images[debut_lot:debut_lot + taille_lot]))))
    scores = np.concatenate(scores)
    predictions = scores.argmax(axis=1)
    perte_moyenne = float(
        np.asarray(optax.softmax_cross_entropy_with_integer_labels(jnp.asarray(scores), jnp.asarray(z))).mean()
    )
    return {
        "loss": perte_moyenne,
        "error": float(np.mean(predictions != z)),
        "predictions": predictions,
    }


def nombre_parametres(machine):
    return sum(feuille.size for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param)))

<a id="affine-mnist"></a>
## 2. Affine score

Vectorise the images and train $x\mapsto Ax+b\in\mathbb R^{10}$. This model has a matrix of $10\times784$ coefficients: each row can be reshaped as an image.

In [ ]:
# Complete this section.

<a id="mlp-mnist"></a>
## 3. MLP

Train an MLP of width 128 with two hidden layers. Like the affine score, it receives a vector of 784 pixels; image geometry is not explicitly imposed on its layers.

In [ ]:
# Complete this section.

<a id="cnn-mnist"></a>
## 4. CNN and parameter sharing

Train a small CNN. The same convolution kernel is applied at every position: this structured matrix therefore shares its parameters spatially. Compare its number of parameters with that of the MLP.

In [ ]:
# Complete this section.

<a id="comparaison-mnist"></a>
## 5. Comparaison

Compare loss, error frequency, training time and number of parameters. Time depends on the computer running the notebook and is therefore not an intrinsic property of the model.

In [ ]:
# Complete this section.

<a id="confusion-mnist"></a>
## 6. Confusion matrix and errors

Plot normalised confusion matrices. Then display some CNN errors and identify the pairs of digits most often confused.

In [ ]:
# Complete this section.

## Summary

Moving from the affine model to the MLP and then the CNN changes neither the score space nor cross-entropy. The function classes and structures imposed on the layers change. MNIST makes this distinction visible without introducing another classifier family.